# `topi.qnn`

In [ ]:

"""Test code for QNN operators."""
import numpy as np
import tvm
from tvm import topi, relay, te
from tvm.contrib import graph_executor
import tvm.topi.testing


def verify_simulated_quantize(data_shape, out_dtype, channels, axis):
    """
    验证模拟量化功能

    该函数用于验证模拟量化操作的正确性，通过比较模拟量化和真实量化的结果
    来确保模拟量化功能的准确性。

    参数:
    data_shape (tuple): 输入数据的形状
    out_dtype (str): 输出数据类型，如'int8', 'uint8', 'int32'
    channels (list): 通道数，用于确定缩放因子和零点的维度
    axis (int): 量化轴

    返回:
    None
    """
    # 创建所有QNN输入的占位符变量
    A = te.placeholder(data_shape, name="value", dtype="float32")
    D = te.placeholder([], name="dtype", dtype="int32")
    S = te.placeholder([te.size_var("scale_dim")], name="scale", dtype="float32")
    Z = te.placeholder([te.size_var("zp_dim")], name="zp", dtype="int32")
    # 创建模拟量化操作
    SIM_Q = topi.nn.simulated_quantize(A, D, output_scale=S, output_zero_point=Z, axis=axis)

    # 创建用于输入的随机numpy值
    a_np = np.random.uniform(size=data_shape).astype("float32")
    d_np = np.int32(topi.nn.SQNN_DTYPE_TO_CODE[out_dtype])  # 将数据类型映射到代码
    s_np = np.random.uniform(low=1e-4, high=0.1, size=channels).astype("float32")  # 缩放因子
    z_np = np.random.uniform(low=-10, high=10, size=channels).astype("int32")  # 零点
    q_np = np.zeros(shape=data_shape, dtype="float32")  # 输出占位符

    def check_target(target, dev):
        """
        在特定目标设备上检查模拟量化的正确性

        参数:
        target: TVM目标设备
        dev: TVM设备上下文

        返回:
        None
        """
        # 将numpy数组包装为tvm.nd数组
        a = tvm.nd.array(a_np, dev)
        d = tvm.nd.array(d_np, dev)
        s = tvm.nd.array(s_np, dev)
        z = tvm.nd.array(z_np, dev)
        q = tvm.nd.array(q_np, dev)

        # 构建等效的relay图
        per_channel = channels[0] != 1  # 判断是否为每通道量化
        a_var = relay.var("a", shape=data_shape, dtype="float32")
        if per_channel:
            s_var = relay.const(s_np)
            z_var = relay.const(z_np)
        else:
            s_var = relay.const(s_np[0])
            z_var = relay.const(z_np[0])
        # 创建真实的QNN量化操作
        real_q_op = relay.qnn.quantize(a_var, s_var, z_var, axis=axis, out_dtype=out_dtype)
        with tvm.transform.PassContext(opt_level=3):
            lib = relay.build(tvm.IRModule.from_expr(real_q_op), target=target)

        # 获取真实的QNN量化输出
        m = graph_executor.GraphModule(lib["default"](dev))
        m.set_input("a", a_np)

        m.run()
        real_q_out = m.get_output(0)

        # 编译模拟量化函数
        with tvm.target.Target(target):
            sched = tvm.topi.testing.get_injective_schedule(target)(SIM_Q)
        func = tvm.build(sched, [A, D, S, Z, SIM_Q], target, name="sim_quantize")
        func(a, d, s, z, q)

        # 检查与真实QNN输出的正确性
        mismatch = q.numpy() != real_q_out.numpy().astype("float32")
        # 允许GPU浮点运算导致的一些舍入误差
        assert np.sum(mismatch) <= 3

    # 在所有启用的目标设备上运行检查
    for target, dev in tvm.testing.enabled_targets():
        check_target(target, dev)


def test_simulated_quantize():
    """
    测试模拟量化功能

    该函数通过调用verify_simulated_quantize函数，在各种输入形状、数据类型、
    通道数和轴配置下测试模拟量化功能的正确性。
    """
    # 测试标量int8量化
    verify_simulated_quantize([1], "int8", [1], -1)
    # 测试2x5形状、轴为1的int8量化
    verify_simulated_quantize([2, 5], "int8", [5], 1)
    # 测试4D张量、轴为-1的int8量化
    verify_simulated_quantize([1, 32, 32, 32], "int8", [32], -1)
    # 测试4D张量、轴为-2的uint8量化
    verify_simulated_quantize([1, 32, 32, 32], "uint8", [32], -2)
    # 测试2x5形状、轴为1的int32量化
    verify_simulated_quantize([2, 5], "int32", [5], 1)


def verify_simulated_dequantize(data_shape, in_dtype, channels, axis):
    """
    验证模拟反量化功能

    该函数用于验证模拟反量化操作的正确性，通过比较模拟反量化和真实反量化的结果
    来确保模拟反量化功能的准确性。

    参数:
    data_shape (tuple): 输入数据的形状
    in_dtype (str): 输入数据类型，如'int8', 'uint8', 'int32'
    channels (list): 通道数，用于确定缩放因子和零点的维度
    axis (int): 反量化轴

    返回:
    None
    """
    # 创建所有QNN输入的占位符变量
    A = te.placeholder(data_shape, name="value", dtype="float32")
    D = te.placeholder([], name="dtype", dtype="int32")
    S = te.placeholder([te.size_var("scale_dim")], name="scale", dtype="float32")
    Z = te.placeholder([te.size_var("zp_dim")], name="zp", dtype="int32")
    # 创建模拟反量化操作
    SIM_DQ = topi.nn.simulated_dequantize(A, D, input_scale=S, input_zero_point=Z, axis=axis)

    # 创建用于输入的随机numpy值
    a_np = np.random.uniform(low=-128, high=127, size=data_shape).astype(in_dtype)
    a_np_f = a_np.astype("float32")  # 转换为float32用于模拟反量化
    d_np = np.int32(topi.nn.SQNN_DTYPE_TO_CODE[in_dtype])  # 将数据类型映射到代码
    s_np = np.random.uniform(low=1e-4, high=0.1, size=channels).astype("float32")  # 缩放因子
    z_np = np.random.uniform(low=-10, high=10, size=channels).astype("int32")  # 零点
    dq_np = np.zeros(shape=data_shape, dtype="float32")  # 输出占位符

    def check_target(target, dev):
        """
        在特定目标设备上检查模拟反量化的正确性

        参数:
        target: TVM目标设备
        dev: TVM设备上下文

        返回:
        None
        """
        # 将numpy数组包装为tvm.nd数组
        a = tvm.nd.array(a_np_f, dev)
        d = tvm.nd.array(d_np, dev)
        s = tvm.nd.array(s_np, dev)
        z = tvm.nd.array(z_np, dev)
        dq = tvm.nd.array(dq_np, dev)

        # 构建等效的relay图
        per_channel = channels[0] != 1  # 判断是否为每通道反量化
        a_var = relay.var("a", shape=data_shape, dtype=in_dtype)
        if per_channel:
            s_var = relay.const(s_np)
            z_var = relay.const(z_np)
        else:
            s_var = relay.const(s_np[0])
            z_var = relay.const(z_np[0])
        # 创建真实的QNN反量化操作
        real_dq_op = relay.qnn.dequantize(a_var, s_var, z_var, axis=axis)
        with tvm.transform.PassContext(opt_level=3):
            lib = relay.build(tvm.IRModule.from_expr(real_dq_op), target=target)

        # 获取真实的QNN反量化输出
        m = graph_executor.GraphModule(lib["default"](dev))
        m.set_input("a", a_np)

        m.run()
        real_dq_out = m.get_output(0)

        # 编译模拟反量化函数
        with tvm.target.Target(target):
            sched = tvm.topi.testing.get_injective_schedule(target)(SIM_DQ)
        func = tvm.build(sched, [A, D, S, Z, SIM_DQ], target, name="sim_quantize")
        func(a, d, s, z, dq)

        # 检查与真实QNN输出的正确性
        tvm.testing.assert_allclose(dq.numpy(), real_dq_out.numpy().astype("float32"), rtol=1e-5)

    for target, dev in tvm.testing.enabled_targets():
        check_target(target, dev)


def test_simulated_dequantize():
    """
    测试模拟反量化功能

    该函数通过调用verify_simulated_dequantize函数，在各种输入形状、数据类型、
    通道数和轴配置下测试模拟反量化功能的正确性。
    """
    # 测试标量int8反量化
    verify_simulated_dequantize([1], "int8", [1], -1)
    # 测试2x5形状、轴为1的int8反量化
    verify_simulated_dequantize([2, 5], "int8", [5], 1)
    # 测试2x5形状、轴为0的int8反量化
    verify_simulated_dequantize([2, 5], "int8", [2], 0)
    # 测试4D张量、轴为-1的int8反量化
    verify_simulated_dequantize([1, 32, 32, 32], "int8", [32], -1)
    # 测试4D张量、轴为-2的uint8反量化
    verify_simulated_dequantize([1, 32, 32, 32], "uint8", [32], -2)
    # 测试2x5形状、轴为1的int32反量化
    verify_simulated_dequantize([2, 5], "int32", [5], 1)


if __name__ == "__main__":
    # 执行所有测试
    test_simulated_quantize()  # 测试模拟量化功能
    test_simulated_dequantize()  # 测试模拟反量化功能
